In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os

# CXL SSD Analysis - Data Visualization

**Quick Configuration**: Edit the cell below to change version and run analyses

In [ ]:
# Metrics mapping (constant for all versions)
METRICS = {
    # --- 次數 (Counts) ---
    "hit_host":           "system.cxl_ssd.readHitsHost",
    "hit_classify":       "system.cxl_ssd.readHitsClassify",
    "hit_store":          "system.cxl_ssd.readHitsStore",
    "hit_dirty":          "system.cxl_ssd.readHitsDirty",
    "large_access":       "system.cxl_ssd.largeAccesses",
    "small_access":       "system.cxl_ssd.smallAccesses",
    "migration_cnode":    "system.cxl_ssd.migrationFromCNode",
    "migration_snode":    "system.cxl_ssd.migrationFromSNode",

    # --- 新增: 總延遲時間組成 (Total Latency Ticks) ---
    "time_total":    "system.cxl_ssd.totalLatency",
    "time_mig_cnode":      "system.cxl_ssd.migrationLatencyFromCNode",
    "time_mig_snode":      "system.cxl_ssd.migrationLatencyFromSNode",
    "time_host":     "system.cxl_ssd.hitsHostLatency",
    "time_classify": "system.cxl_ssd.hitsClassifyLatency",
    "time_store":    "system.cxl_ssd.hitsStoreLatency",
    "time_dirty":    "system.cxl_ssd.hitsDirtyLatency",
    "time_large":    "system.cxl_ssd.largeAccessLatency",
    "time_small":    "system.cxl_ssd.smallAccessLatency" # 對應 Flash Access
}

print(f"✅ Metrics mapping loaded ({len(METRICS)} metrics)")


✅ Metrics mapping loaded (16 metrics)


In [ ]:
def parse_stats(folder_path):
    stats_file = os.path.join(folder_path, "stats.txt")
    data = {k: 0.0 for k in METRICS.keys()} 
    
    if not os.path.exists(stats_file):
        print(f"⚠️  找不到 {stats_file}")
        return data

    try:
        with open(stats_file, "r") as f:
            for line in f:
                parts = line.split()
                if not parts: continue
                name = parts[0]
                for key, metric_name in METRICS.items():
                    if name == metric_name:
                        try:
                            data[key] = float(parts[1])
                        except:
                            pass
        return data
    except Exception as e:
        print(f"❌ 讀取錯誤: {e}")
        return data

In [ ]:
# Helper function for stacked bars
def plot_stacked_bar(ax, data_dict, labels, title, ylabel, is_normalized=False, colors=None):
    bottom_val = np.zeros(len(labels))
    for key, data in data_dict.items():
        ax.bar(labels, data, bottom=bottom_val, label=key, color=colors[key])
        bottom_val += data
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    if is_normalized:
        ax.set_ylim(0, 1)
    else:
        ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

In [ ]:
# ==========================================
# 📊 Load and Process Data
# ==========================================

def load_all_results(experiments):
    """Load stats from all experiment folders"""
    return {label: parse_stats(folder) for label, folder in experiments}

In [ ]:
# ==========================================
# Figure 1: Full Analysis (4 subplots)
# ==========================================

def create_full_analysis_figure(results, labels, version, output_file):
    """Generate comprehensive 4-panel analysis figure"""
    fig, axs = plt.subplots(2, 2, figsize=(16, 16))
    (ax1, ax2), (ax3, ax4) = axs

    # Extract data
    hit_host = np.array([results[l]["hit_host"] for l in labels])
    hit_internal = np.array([results[l]["hit_classify"] + results[l]["hit_store"] + results[l]["hit_dirty"] for l in labels])
    miss_flash = np.array([results[l]["miss_flash"] for l in labels])
    
    t_host = np.array([results[l]["time_host"] for l in labels])
    t_internal = np.array([results[l]["time_classify"] + results[l]["time_store"] + results[l]["time_dirty"] for l in labels])

    # --- Migration counts: support both split (migration_cnode/migration_snode)
    # and combined (`migration` or `migrations`) metric names
    migrations = []
    for l in labels:
        r = results[l]
        mig = r.get("migrations")
        if mig is None:
            mig = r.get("migration")
        if mig is None:
            mig = r.get("migration_cnode", 0.0) + r.get("migration_snode", 0.0)
        migrations.append(mig)

    # --- Migration latency: support combined `time_mig` or split `time_mig_cnode`/`time_mig_snode`
    t_mig_cnode = np.array([results[l].get("time_mig_cnode", 0.0) for l in labels])
    t_mig_snode = np.array([results[l].get("time_mig_snode", 0.0) for l in labels])
    # If combined metric present, use it; otherwise fall back to sum of split parts
    t_mig_comb = np.array([results[l].get("time_mig", t_mig_cnode[i] + t_mig_snode[i]) for i, l in enumerate(labels)])
    # Decide whether we have split components (non-zero) or only combined
    has_split = np.any(t_mig_cnode) or np.any(t_mig_snode)
    if has_split:
        t_mig_cnode_arr = t_mig_cnode
        t_mig_snode_arr = t_mig_snode
    else:
        t_mig_cnode_arr = t_mig_comb
        t_mig_snode_arr = np.zeros_like(t_mig_comb)

    t_flash = np.array([results[l]["time_small"]+results[l]["time_large"] for l in labels])
    
    # Panel 1: Access Breakdown
    ax1.bar(labels, hit_host, label='Host Hit (DRAM)', color='#2ca02c', alpha=0.8, width=0.5)
    ax1.bar(labels, hit_internal, bottom=hit_host, label='Internal Hit', color='#ff7f0e', alpha=0.8, width=0.5)
    ax1.bar(labels, miss_flash, bottom=hit_host+hit_internal, label='Flash Miss', color='#d62728', alpha=0.8, width=0.5)
    ax1.set_title("Request Count Breakdown\n(Where are requests served?)", fontweight='bold')
    ax1.set_ylabel("Count")
    ax1.legend()
    ax1.grid(axis='y', linestyle='--', alpha=0.3)
    
    # Panel 2: Migration Activity
    bars2 = ax2.bar(labels, migrations, color=['#1f77b4', '#9467bd'], width=0.5, alpha=0.9)
    ax2.set_title("Migration Events\n(Pages moved SSD -> DRAM)", fontweight='bold')
    ax2.set_ylabel("Count")
    ax2.set_yscale('log')
    ax2.grid(axis='y', linestyle='--', alpha=0.3)
    for bar in bars2:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., h, f'{int(h):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Panel 3: Latency Distribution
    x = np.arange(len(labels))
    width = 0.35
    val_fast = [results[l]["lat_fast"] for l in labels]
    val_slow = [results[l]["lat_slow"] for l in labels]
    ax3.bar(x - width/2, val_fast, width, label='Fast (<4µs)', color='#2ca02c', alpha=0.8)
    ax3.bar(x + width/2, val_slow, width, label='Slow (~100µs)', color='#d62728', alpha=0.8)
    ax3.set_title("Latency Count Distribution\n(Fast Hits vs Slow Misses)", fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels(labels)
    ax3.set_ylabel("Count")
    ax3.set_yscale('log')
    ax3.legend()
    ax3.grid(axis='y', linestyle='--', alpha=0.3)

    # Panel 4: Total Latency Breakdown
    ax4.bar(labels, t_host, label='Host Hit Time', color='#2ca02c', alpha=0.8, width=0.5)
    ax4.bar(labels, t_internal, bottom=t_host, label='Internal Hit Time', color='#ff7f0e', alpha=0.8, width=0.5)
    ax4.bar(labels, t_mig_cnode_arr, bottom=t_host+t_internal, label='Migration Overhead (CNode)', color='#9467bd', alpha=0.8, width=0.5)
    ax4.bar(labels, t_mig_snode_arr, bottom=t_host+t_internal+t_mig_cnode_arr, label='Migration Overhead (SNode)', color="#6795bd", alpha=0.8, width=0.5)
    ax4.bar(labels, t_flash, bottom=t_host+t_internal+t_mig_cnode_arr+t_mig_snode_arr, label='Flash Access Time', color='#d62728', alpha=0.8, width=0.5)
    ax4.set_title("Total System Time Breakdown\n(Where is time spent?)", fontweight='bold')
    ax4.set_ylabel("Total Latency (Ticks)")
    ax4.legend(loc='upper left', fontsize='small') 
    ax4.grid(axis='y', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_file)
    print(f"✅ Saved: {output_file}")
    # plt.show()


In [ ]:
# ==========================================
# 📊 Figure 2: Response Distribution Analysis
# ==========================================

def create_distribution_figure(results, labels, version, output_file):
    """Generate 4-panel response distribution analysis"""
    
    # Extract and process data
    counts_host     = np.array([results[l]["hit_host"] for l in labels])
    counts_classify = np.array([results[l]["hit_classify"] for l in labels])
    counts_store    = np.array([results[l]["hit_store"] for l in labels])
    counts_dirty    = np.array([results[l]["hit_dirty"] for l in labels])
    counts_flash    = np.array([results[l]["miss_flash"] for l in labels])

    counts_total = counts_host + counts_classify + counts_store + counts_dirty + counts_flash
    counts_total_safe = np.where(counts_total == 0, 1, counts_total)
    
    counts_no_flash = counts_host + counts_classify + counts_store + counts_dirty
    counts_no_flash_safe = np.where(counts_no_flash == 0, 1, counts_no_flash)

    # Define colors
    colors = {
        'host':     '#2ca02c',
        'classify': '#ff7f0e',
        'store':    '#9467bd',
        'dirty':    '#8c564b',
        'flash':    '#d62728'
    }

    fig, axs = plt.subplots(2, 2, figsize=(16, 16))

    # Panel 1: Absolute (with Flash)
    data_abs_all = {
        'host': counts_host,
        'classify': counts_classify,
        'store': counts_store,
        'dirty': counts_dirty,
        'flash': counts_flash
    }
    plot_stacked_bar(axs[0, 0], data_abs_all, labels, "Total Requests (Absolute)", "Count", colors=colors)
    axs[0, 0].legend(loc='upper left', fontsize='small', framealpha=0.8)

    # Panel 2: Normalized (with Flash)
    p_host = counts_host / counts_total_safe
    p_classify = counts_classify / counts_total_safe
    p_store = counts_store / counts_total_safe
    p_dirty = counts_dirty / counts_total_safe
    p_flash = counts_flash / counts_total_safe
    
    data_norm_all = {
        'host': p_host,
        'classify': p_classify,
        'store': p_store,
        'dirty': p_dirty,
        'flash': p_flash
    }
    plot_stacked_bar(axs[0, 1], data_norm_all, labels, "Distribution (Normalized)", "Fraction", is_normalized=True, colors=colors)
    axs[0, 1].legend(loc='upper left', fontsize='small', framealpha=0.8)

    # Panel 3: Absolute (without Flash)
    data_abs_nf = {
        'host': counts_host,
        'classify': counts_classify,
        'store': counts_store,
        'dirty': counts_dirty
    }
    plot_stacked_bar(axs[1, 0], data_abs_nf, labels, "RAM Hits (Absolute w/o Flash)", "Count (RAM only)", colors=colors)

    # Panel 4: Normalized (without Flash)
    p_nf_host = counts_host / counts_no_flash_safe
    p_nf_classify = counts_classify / counts_no_flash_safe
    p_nf_store = counts_store / counts_no_flash_safe
    p_nf_dirty = counts_dirty / counts_no_flash_safe
    
    data_norm_nf = {
        'host': p_nf_host,
        'classify': p_nf_classify,
        'store': p_nf_store,
        'dirty': p_nf_dirty
    }
    plot_stacked_bar(axs[1, 1], data_norm_nf, labels, "RAM Distribution (Normalized w/o Flash)", "Fraction (RAM only)", is_normalized=True, colors=colors)

    plt.tight_layout()
    plt.savefig(output_file)
    print(f"✅ Saved: {output_file}")


In [ ]:
# ==========================================
# F4C8 Detailed Statistics Summary
# ==========================================

def print_statistics_summary(results, labels, version, output_file, save_to_file=False):
    """Print detailed breakdown of hits, misses, and latency"""
    
    output_lines = []
    output_lines.append(f"\n{'='*70}")
    output_lines.append(f"📊 STATISTICS SUMMARY - VERSION {version}")
    output_lines.append(f"{'='*70}\n")
    
    for label in labels:
        r = results[label]
        
        # Request counts
        r_host = r["hit_host"]
        r_classify = r["hit_classify"]
        r_store = r["hit_store"]
        r_dirty = r["hit_dirty"]
        r_flash = r["miss_flash"]
        # migrations: support combined or split names
        r_mig = r.get("migrations", None)
        if r_mig is None:
            r_mig = r.get("migration", r.get("migration_cnode", 0.0) + r.get("migration_snode", 0.0))
        
        total_req = r_host + r_classify + r_store + r_dirty + r_flash
        if total_req == 0:
            total_req = 1
        
        # Time breakdown: support combined or split names
        t_host = r.get("time_host", 0.0)
        t_classify = r.get("time_classify", 0.0)
        t_store = r.get("time_store", 0.0)
        t_dirty = r.get("time_dirty", 0.0)
        t_mig = r.get("time_mig", r.get("time_mig_cnode", 0.0) + r.get("time_mig_snode", 0.0))
        t_flash = r.get("time_small", 0.0) + r.get("time_large", 0.0)
        
        total_time = t_host + t_classify + t_store + t_dirty + t_mig + t_flash
        if total_time == 0:
            total_time = 1
        
        output_lines.append(f"{'─'*70}")
        output_lines.append(f"🔹 {label}")
        output_lines.append(f"{'─'*70}")
        
        output_lines.append(f"\n📍 Request Distribution:")
        output_lines.append(f"   Host Hits:     {int(r_host):>12,}  ({r_host/total_req*100:>6.2f}%)")
        output_lines.append(f"   Classify Hits: {int(r_classify):>12,}  ({r_classify/total_req*100:>6.2f}%)")
        output_lines.append(f"   Store Hits:    {int(r_store):>12,}  ({r_store/total_req*100:>6.2f}%)")
        output_lines.append(f"   Dirty Hits:    {int(r_dirty):>12,}  ({r_dirty/total_req*100:>6.2f}%)")
        output_lines.append(f"   Flash Misses:  {int(r_flash):>12,}  ({r_flash/total_req*100:>6.2f}%)")
        output_lines.append(f"   {'─'*50}")
        output_lines.append(f"   Total:         {int(total_req):>12,}  (100.00%)")
        output_lines.append(f"   Migrations:    {int(r_mig):>12,}")
        
        output_lines.append(f"\n⏱️  Time Breakdown (Ticks):")
        output_lines.append(f"   Host Time:     {int(t_host):>12,}  ({t_host/total_time*100:>6.2f}%)")
        output_lines.append(f"   Classify Time: {int(t_classify):>12,}  ({t_classify/total_time*100:>6.2f}%)")
        output_lines.append(f"   Store Time:    {int(t_store):>12,}  ({t_store/total_time*100:>6.2f}%)")
        output_lines.append(f"   Dirty Time:    {int(t_dirty):>12,}  ({t_dirty/total_time*100:>6.2f}%)")
        output_lines.append(f"   Migration Time:{int(t_mig):>12,}  ({t_mig/total_time*100:>6.2f}%)")
        output_lines.append(f"   Flash Time:    {int(t_flash):>12,}  ({t_flash/total_time*100:>6.2f}%)")
        output_lines.append(f"   {'─'*50}")
        output_lines.append(f"   Total:         {int(total_time):>12,}  (100.00%)")
        output_lines.append("")
    
    # Print to console
    output_text = "\n".join(output_lines)
    print(output_text)
    
    # Save to file if requested
    if save_to_file:
        with open(output_file, 'w') as f:
            f.write(output_text)
        print(f"✅ Statistics saved to: {output_file}")
    
    return output_text


In [4]:
def create_analysis_figure(results, labels, version, output_file):
    """Generate analysis figure"""

    

In [2]:
# ==========================================
# 📋 CONFIGURATION - EDIT THIS SECTION ONLY
# ==========================================

VERSION = "v2"  # ← Change this to switch versions
MODE = "ycsb"

# Experiment definitions (these use the VERSION above)
EXPERIMENTS = [
    ("Linear (Bi-Tiered)", f"m5out_{VERSION}_{MODE}_linear_256"),
    ("Random (Bi-Tiered)", f"m5out_{VERSION}_{MODE}_random_256"),
    ("Linear (Baseline)", f"m5out_{VERSION}_{MODE}_linear_0"),
    ("Random (Baseline)", f"m5out_{VERSION}_{MODE}_random_0")
]

full_analysis_filename = f"cxl_{VERSION}_{MODE}_full_analysis.png"
distribution_filename = f"cxl_{VERSION}_{MODE}_response_distribution.png"
statistics_filename = f"cxl_{VERSION}_{MODE}_detailed_statistics.txt"


print(f"✅ Configuration loaded for VERSION: {VERSION}")
print(f"   Experiments: {len(EXPERIMENTS)}")


✅ Configuration loaded for VERSION: v2
   Experiments: 4


In [ ]:
# Load data based on current EXPERIMENTS and METRICS
results = load_all_results(EXPERIMENTS, METRICS)
labels = list(results.keys())

print(f"✅ Loaded {len(labels)} experiments")
for label in labels:
    print(f"   - {label}")

print("\n" + "="*70)
print("📊 Running all analyses...")
print("="*70 + "\n")
    
# Run all analyses and save statistics
create_full_analysis_figure(results, labels, VERSION, full_analysis_filename)
create_distribution_figure(results, labels, VERSION, distribution_filename)
# print_statistics_summary(results, labels, version=VERSION, output_file=statistics_filename, save_to_file=True)

print("\n✅ All analyses complete!")


✅ Loaded 4 experiments
   - Linear (Bi-Tiered)
   - Random (Bi-Tiered)
   - Linear (Baseline)
   - Random (Baseline)

📊 Running all analyses...



/tmp/ipykernel_249186/998333421.py:66: UserWarning: Tight layout not applied. tight_layout cannot make Axes height small enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ Saved: cxl_v1_nvm_full_analysis.png
✅ Saved: cxl_v1_nvm_response_distribution.png

📊 STATISTICS SUMMARY - VERSION v1

──────────────────────────────────────────────────────────────────────
🔹 Linear (Bi-Tiered)
──────────────────────────────────────────────────────────────────────

📍 Request Distribution:
   Host Hits:      399,677,661  ( 94.05%)
   Classify Hits:   17,869,024  (  4.20%)
   Store Hits:               0  (  0.00%)
   Dirty Hits:       6,401,991  (  1.51%)
   Flash Misses:     1,014,691  (  0.24%)
   ──────────────────────────────────────────────────
   Total:          424,963,367  (100.00%)
   Migrations:       1,014,691

⏱️  Time Breakdown (Ticks):
   Host Time:     31,974,212,880,000  ( 23.73%)
   Classify Time: 286,644,021,000  (  0.21%)
   Store Time:               0  (  0.00%)
   Dirty Time:    640,199,100,000  (  0.48%)
   Migration Time:286,644,021,000  (  0.21%)
   Flash Time:    101,570,569,100,000  ( 75.37%)
   ─────────────────────────────────────────────────